# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets
from pprint import pprint

print("Available record sets (by @id):")
record_sets = []
for rs in metadata.record_sets:
    print(f"  - {rs['@id']}: {rs.get('name', '(no name given)')}")
    record_sets.append(rs['@id'])

# Show fields for the first record set
if record_sets:
    selected_record_set_id = record_sets[0]
    selected_rs = next(rs for rs in metadata.record_sets if rs['@id'] == selected_record_set_id)
    print(f"\nFields for record set {selected_record_set_id}:")
    for field in selected_rs.get('fields', []):
        field_id = field['@id']
        field_name = field.get('name', field_id)
        print(f"  - {field_id}: {field_name}")
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from the available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract records from each record set into a DataFrame
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Fields (columns): {df.columns.tolist()}")
    except Exception as e:
        print(f"  Error loading records: {e}")

# For demonstration, select the first record set (if available)
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"\nSample data from {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping. Adjust the fields using their `@id` for reference.

In [ ]:
# Identify a numeric field for demonstration, e.g. a peptide count or abundance column
numeric_field_id = None
df = dataframes.get(main_record_set_id)

if df is not None:
    # Try auto-detect numeric field candidates by data type
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric field detected. Please modify `numeric_field_id` variable below to match your data.")

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'bool' else 1
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try a group field (categorical)
    group_field = None
    # Try to auto-detect a suitable categorical field
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field_id:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field}: (top 5)")
        display(grouped_df.head())
    else:
        print("No categorical field found for grouping.")
else:
    print("No numeric field selected.")

## 5. Visualization
Visualize the distribution of the numeric field, or a group comparison if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and df is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Cannot plot: no numeric field found.")

## 6. Conclusion
This notebook demonstrated how to load, examine, filter, and visualize data from a Croissant dataset using the `mlcroissant` library. 
- We explored the dataset, identified record sets, inspected fields, and loaded tabular data.
- Simple EDA such as filtering and normalization was performed.
- Histograms and groupwise comparisons provided insight into the data distribution.

Continue analysis with advanced modeling or domain-specific preprocessing as needed.